# dim_channel_scd2 (SCD 2, poc `scd_type_2_new` method)

One row per `channel_id` × version. Built like poc `dim_family`.

**SCD columns (major → new version):** `channel_title`, `custom_url`, `country`. Everything else is minor (updated in place on the current row).

Version dates: first version `start_date = created_at`; a major change closes the current row (`current_flag = 'N'`, `end_date = modified_at`) and opens a new one (`start_date = modified_at`, `end_date = 9999-12-31 23:59:59`, `current_flag = 'Y'`). Not in any job yet.

In [ ]:
%run ./_scd

# Set up environment

In [ ]:
for layer in ("silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
dbutils.widgets.dropdown("load_type", "incremental", ["incremental", "full"])
dbutils.widgets.text("from_date", "")   # yyyy-MM-dd, Vietnam day, inclusive; empty = yesterday
dbutils.widgets.text("to_date", "")     # exclusive; empty = tomorrow

silver = dbutils.widgets.get("silver")
gold = dbutils.widgets.get("gold")
load_type = dbutils.widgets.get("load_type")
from_date, to_date = get_incremental_dates(dbutils.widgets.get("from_date"), dbutils.widgets.get("to_date"))
print(f"load_type = {load_type}, window = [{from_date}, {to_date})")

scd_cols = ["channel_title", "custom_url", "country"]

# Read Source Table

In [ ]:
stg_channels = spark.table(f"{silver}.stg_channels")

## Incremental: reads in the window, then the last read per channel

In [ ]:
from pyspark.sql.window import Window

if load_type == "incremental":
    stg_channels = stg_channels.filter((F.col("snapshot_date") >= from_date) & (F.col("snapshot_date") < to_date))

last_read_first = Window.partitionBy("channel_id").orderBy(F.col("snapshot_ts").desc(), F.col("etl_date").desc_nulls_last())
channels = stg_channels.withColumn("rn", F.row_number().over(last_read_first)).filter("rn = 1")

# Transformation

In [ ]:
result_df = channels.select(
    F.col("channel_id").cast("string").alias("channel_id"),
    F.col("channel_title").cast("string").alias("channel_title"),
    F.col("channel_description").cast("string").alias("channel_description"),
    F.col("custom_url").cast("string").alias("custom_url"),
    F.col("country").cast("string").alias("country"),
    F.col("default_language").cast("string").alias("default_language"),
    F.col("thumbnail_url").cast("string").alias("thumbnail_url"),
    F.col("published_at").cast("timestamp").alias("published_at"),
    F.col("created_at").cast("timestamp").alias("created_at"),
    F.col("snapshot_ts").cast("timestamp").alias("modified_at"),
)

## Add metadata

In [ ]:
insert_date = datetime.now()
etl_by = "Databricks"
result_df = result_df.withColumn("etl_date", F.lit(insert_date)).withColumn("etl_by", F.lit(etl_by))

## Add checksums

In [ ]:
primary_key = ["channel_id"]
result_df = add_scd_checksum(result_df, scd_cols)
result_df = add_checksums(result_df, primary_key, exclude=["created_at", "modified_at"])

## Create table

`id` = surrogate key per version (IDENTITY).

In [ ]:
path = f"{gold}.dim_channel_scd2"
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {path} (
        id BIGINT GENERATED BY DEFAULT AS IDENTITY,
        channel_id STRING, channel_title STRING, channel_description STRING, custom_url STRING,
        country STRING, default_language STRING, thumbnail_url STRING, published_at TIMESTAMP,
        created_at TIMESTAMP, modified_at TIMESTAMP, etl_date TIMESTAMP, etl_by STRING,
        primary_checksum STRING, record_checksum STRING, scd_checksum STRING,
        current_flag STRING, start_date TIMESTAMP, end_date TIMESTAMP
    ) USING DELTA
    COMMENT 'Channel dimension, SCD 2 (poc scd_type_2_new method). id = surrogate key per version, -1 = Unknown.'
""")

# Write to Table

`scd_type_2` (helper `_scd`, same steps as poc `scd_type_2_new`): minor → update the current row; major → close it + insert a new version; new key → insert.

In [ ]:
counts = scd_type_2(result_df, path)
print(f"minor (update in place): {counts['minor']} · major (new version): {counts['major']} · new channels: {counts['new']}")

## Unknown row

In [ ]:
if spark.table(path).where("id = -1").isEmpty():
    spark.sql(f"""
        INSERT INTO {path} (id, channel_title, etl_date, etl_by, current_flag, start_date, end_date)
        VALUES (-1, 'Unknown', current_timestamp(), 'Databricks', 'Y', TIMESTAMP '1900-01-01', TIMESTAMP '{MAX_DATE}')
    """)

## Check: one current row per channel, no start_date > end_date

In [ ]:
c = spark.sql(f"""
    SELECT COUNT(*) AS n,
           COUNT_IF(current_flag = 'Y') AS cur,
           COUNT(DISTINCT CASE WHEN current_flag = 'Y' THEN channel_id END) AS cur_ids,
           COUNT_IF(start_date > end_date) AS bad_dates
    FROM {path} WHERE channel_id IS NOT NULL
""").first()
assert c["cur"] == c["cur_ids"], f"{path}: a channel has more than one current row"
assert c["bad_dates"] == 0, f"{path}: {c['bad_dates']} rows with start_date > end_date"
print(f"✅ {path}: {c['n']} versions, {c['cur']} current channels")